In [1]:
# llm libraries
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import pandas as pd
import time

/home/dferna3/.conda/envs/env_ft_llama2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print('torch version:', torch.__version__)
print('CUDA version', torch.version.cuda)
print('CUDA is available:', torch.cuda.is_available())
print('Number of GPUs available:', torch.cuda.device_count())

torch version: 2.3.0
CUDA version 12.1
CUDA is available: True
Number of GPUs available: 4


In [ ]:
file_path = "../data/"
results_filename = "../data/results.csv"
latency_filename = "../data/latency_results.csv"

In [4]:
def export_model(file_name, data, headers):
    with open(file_name, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(headers)  # Write the header row
        writer.writerows(data)  # Write all rows of data

In [5]:
models = [
    #"meta-llama/Llama-3.2-1B-Instruct",
    #"meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    #"meta-llama/Meta-Llama-3-70B-Instruct"
]

In [6]:
# HuggingFace configuration
config_data = json.load(open(file_path+'/llama_config.json'))
HF_TOKEN = config_data['HF_TOKEN']

In [7]:
prompt_instructions = """
You are an AI driving analysis tool. Your primary function is to assess whether a given driving scenario is consistent or exhibits signs of an adversarial attack (e.g., tampered traffic signs). You have expert knowledge of traffic regulations, road signage, and common driving practices.

Given a driving scenario in JSON format, your goal is to:
    1. Determine if there is any inconsistency in the scenario (e.g., a sign or speed limit that conflicts with typical real-world traffic laws).
    2. Provide a clear and concise explanation of your reasoning, identifying which specific part of the scenario is inconsistent and referencing relevant traffic rules or logical deductions.

Use the following exact response format. Fill in the placeholders appropriately:

    Inconsistency Detected: [Yes/No]
    Reasoning: [Provide a thorough and concise explanation. Keep it to 3-5 sentences and ensure it demonstrates your reasoning.]

Follow a step-by-step reasoning process internally but only provide the final output in the specified format. Be precise and adhere strictly to the format.
"""

In [8]:
# Load tokenizer
#tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

In [9]:
#model = AutoModelForCausalLM.from_pretrained(model_name,
#                                         cache_dir = "/scratch/dferna3",
#                                         #device_map = 'cuda',
#                                         device_map = "auto",
#                                         #device_map = "balanced",
#                                         #device_map = "balanced_low_0",
#                                         #quantization_config = bnb_config,
#                                         #attn_implementation="flash_attention_2",
#                                         token=HF_TOKEN)

In [10]:
def load_scenarios(file_path):
    with open(file_path, "r") as file:
        scenarios = json.load(file)
    return scenarios["Scenarios"]

In [11]:
scenario_filepath = file_path + "scenarios.json"
scenarios = load_scenarios(scenario_filepath)

In [12]:
outputs = []
latency_outputs = []

In [ ]:
for model_name in models:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
    
    model = AutoModelForCausalLM.from_pretrained(model_name,
                                             cache_dir = "",
                                             #device_map = 'cuda',
                                             device_map = "auto",
                                             #device_map = "balanced",
                                             #device_map = "balanced_low_0",
                                             #quantization_config = bnb_config,
                                             #attn_implementation="flash_attention_2",
                                             token=HF_TOKEN)
    text_generator = pipeline(
        'text-generation',
        model=model_name,
        max_new_tokens=500,
        return_full_text=False,
        tokenizer = tokenizer,
        temperature = 0.8,
        top_p = 0.9
        )

    for scenario in scenarios:
        scenario_input =(
            f"Scenario ID: {scenario['id']}\n"
            f"Road Type: {scenario['RoadType']}\n"
            f"Speed: {scenario['Speed']} mph\n"
            f"Lane Markings: {', '.join(scenario['LaneMarkings'])}\n"
            f"Traffic Signs: {', '.join(scenario['TrafficSigns'])}\n"
            f"{', '.join([str(device) for device in scenario['TrafficControlDevices']]) if scenario['TrafficControlDevices'] else 'None'}\n"
            f"Time of Day: {scenario['TimeOfDay']}\n"
            f"Description: {scenario['Description']}\n"
        )
        print(f"******************************************************************************************")
        print(f"For Model: {model_name} - Processing Scenario {scenario['id']}...")
        start_time = time.time()

        response = text_generator(f"{prompt_instructions}\n{scenario_input}")

        elapsed_time = round(time.time() - start_time, 4)
        # Pretty print the generated response
        generated_text = response[0]['generated_text']
        print(generated_text)
        print()
        outputs.append(generated_text)
        latency_outputs.append(elapsed_time)
    
#    ## Output results
#    # read current csv file 
#    current_results_df = pd.read_csv(results_filename)
#    scenario_id = "scenario_id"
#    
#    new_results_df = pd.DataFrame({
#        scenario_id: [scenario["id"] for scenario in scenarios],
#        model_name: outputs
#    })
#
#    # Merge new model's ouputs with existing DataFrames
#    updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")
#
#    updated_df.to_csv(results_filename, index=False)
#
#    
#    ## Output Latency results
#    #read current latency csv file 
#    current_latency_results_df = pd.read_csv(latency_filename)
#
#    new_latency_df = pd.DataFrame({
#        scenario_id: [scenario["id"] for scenario in scenarios],
#        f"latency_{model_name}": latency_outputs
#
#    })
#
#    # Merge new model's ouputs with existing DataFrames
#    updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")
#
#    updated_latency_df.to_csv(latency_filename, index=False)
#
#    del model
#    del tokenizer

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.78s/it]
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


******************************************************************************************
For Model: meta-llama/Llama-3.1-8B-Instruct - Processing Scenario 1...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


JSON Representation:
```json
{
  "scenario_id": "1",
  "road_type": "Highway",
  "speed": "65",
  "lane_markings": "Broken white line",
  "traffic_signs": ["R1-1: Stop Sign"],
  "time_of_day": "Daytime",
  "description": "A car traveling at 65 mph on a highway encounters a stop sign placed on the roadway shoulder."
}
```

### Step 1: Determine the consistency of the speed limit with typical real-world traffic laws.
Speed limits on highways are typically higher than 65 mph in many jurisdictions, suggesting this speed limit might be too low for a highway.

### Step 2: Evaluate the placement and consistency of the stop sign with typical real-world traffic laws.
A stop sign placed on the roadway shoulder of a highway, especially where the speed limit is higher, does not align with standard traffic placement practices. Stop signs are usually placed at intersections or where traffic flow must yield to other traffic or pedestrians, not on the shoulder of a highway.

### Step 3: Assess the con

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Other Traffic: None

JSON Representation:
```
{
  "road_type": "Urban Intersection",
  "speed_limit": 30,
  "lane_markings": "Double yellow center line",
  "traffic_signs": [
    {
      "type": "R1-1",
      "description": "Stop Sign"
    },
    {
      "type": "F-2",
      "description": "Flashing Beacon"
    }
  ],
  "time_of_day": "Evening",
  "other_traffic": "None"
}
```

### Step 1: Identify the key elements of the scenario
- Road type: Urban Intersection
- Speed limit: 30 mph
- Lane markings: Double yellow center line
- Traffic signs: Stop sign (R1-1) at cross street with a flashing beacon
- Time of day: Evening
- Other traffic: None

### Step 2: Analyze the traffic signs
- The stop sign (R1-1) is correctly placed at a cross street to indicate drivers must come to a complete stop before proceeding.
- The flashing beacon (F-2) enhances the visibility of the stop sign, especially in low-light conditions, which is appropriate for an evening setting.

### Step 3: Evaluate the consi

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


```json
{
  "road_type": "two-lane_residential",
  "speed": 25,
  "lane_markings": {
    "right": "solid_white",
    "center": "broken_yellow"
  },
  "traffic_signs": [
    {
      "speed_limit": 20,
      "activation": "flashing_beacon",
      "description": "School Speed Limit"
    }
  ],
  "time_of_day": "morning",
  "description": "A vehicle approaches a school zone in a neighborhood during morning hours with a sign showing a reduced speed limit of 20 mph activated by a flashing beacon."
}
```

Inconsistency Detected: [Yes/No]
Reasoning: [Provide a thorough and concise explanation. Keep it to 3-5 sentences and ensure it demonstrates your reasoning.]```json
{
  "Inconsistency Detected": "Yes",
  "Reasoning": "The scenario presents an inconsistency because the speed limit reduces to 20 mph when the flashing beacon is activated, which is a standard practice for school zones. However, the vehicle is approaching the school zone at a speed of 25 mph, which is higher than the reduced spee

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


```python
def analyze_driving_scenario(scenario):
    """
    Analyze a driving scenario for inconsistencies.
    
    Args:
        scenario (dict): A dictionary containing information about the driving scenario.
    
    Returns:
        str: Inconsistency detected (Yes/No) and a concise explanation of the reasoning.
    """

    # Initialize variables
    road_type = scenario['Road Type']
    speed = scenario['Speed']
    lane_markings = scenario['Lane Markings']
    traffic_signs = scenario['Traffic Signs']
    time_of_day = scenario['Time of Day']

    # Check for inconsistency based on traffic rules
    if road_type == 'Rural Two-Lane Road' and speed == 45 and 'School Speed Limit' in traffic_signs and traffic_signs['School Speed Limit'] == 30:
        return "Inconsistency Detected: Yes\nReasoning: The speed limit is inconsistent with typical rural two-lane road speed limits and the school speed limit when the beacon is flashing. According to traffic regulations, school speed lim

In [ ]:
len(outputs)

9

In [ ]:
# read current csv file 
current_results_df = pd.read_csv(results_filename)
scenario_id = "scenario_id"

new_results_df = pd.DataFrame({
    scenario_id: [scenario["id"] for scenario in scenarios],
    model_name: outputs
})

# Merge new model's ouputs with existing DataFrames
updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")

updated_df.to_csv(results_filename, index=False)


## Output Latency results
#read current latency csv file 
current_latency_results_df = pd.read_csv(latency_filename)

new_latency_df = pd.DataFrame({
    scenario_id: [scenario["id"] for scenario in scenarios],
    f"latency_{model_name}": latency_outputs

})

# Merge new model's ouputs with existing DataFrames
updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")

updated_latency_df.to_csv(latency_filename, index=False)

del model
del tokenizer

ValueError: All arrays must be of the same length

In [ ]:
#scenario_id = "scenario_id"
#csv_headers = [scenario_id, model]
## read current csv file 
#current_results_df = pd.read_csv(results_filename)
#
#new_results_df = pd.DataFrame({
#    scenario_id: [scenario["id"] for scenario in scenarios],
#    model_name: outputs
#})
#
## Merge new model's ouputs with existing DataFrames
#updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")
#
#updated_df.to_csv(results_filename, index=False)
#

In [ ]:
#csv_headers = [scenario_id, f"latency_{model_name}"]
#
##read current latency csv file 
#current_latency_results_df = pd.read_csv(latency_filename)
#
#new_latency_df = pd.DataFrame({
#    scenario_id: [scenario["id"] for scenario in scenarios],
#    f"latency_{model_name}": latency_outputs
#    
#})
#
## Merge new model's ouputs with existing DataFrames
#updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")
#
#updated_latency_df.to_csv(latency_filename, index=False)

In [ ]:
for output in outputs:
    print()
    print("*************************")
    print(output)